# MCE–ILCADT v4 — Tahap 5: run utama dan analisis sensitivitas (Colab)

Paket ini berisi kode yang telah diaudit pada Tahap 4, dataset master v4 terpseudonimisasi (tanpa NIK, nama, alamat, dan tanpa salt), serta SAP v4 yang dibekukan.

**Aturan**
1. Pilih runtime **CPU** (Runtime → Ubah jenis runtime → CPU). Jalankan sel 1, 2, 2b, dan 3 di awal setiap sesi Colab. Sel 2b memastikan lingkungan sama dengan paper 1 (Python 3.12, TensorFlow 2.20.0, scikit-learn 1.6.1, XGBoost 3.1.2, LightGBM 4.6.0, CatBoost 1.2.8, tanpa GPU).
2. Jalankan satu sel sesi (S1, S2, …) per sesi. Setiap sel bisa diulang bila runtime terputus: horizon yang sudah selesai dilewati, horizon yang sudah dibekukan hanya menjalankan tahap (f).
3. Jangan mengubah file di `code/` atau `data/`. Kode dan data diperiksa hash-nya; tahap (f) menolak berjalan bila artefak beku berubah.
4. Hasil uji tidak dicetak di layar. Setelah semua sesi selesai, jalankan sel paket hasil dan unggah zip-nya ke chat.

Perkiraan waktu (CPU Colab): ±35–45 menit per horizon.

In [ ]:
# 1. Mount Google Drive dan lokasi paket
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/MCE_FP_v4'   # folder hasil ekstrak MCE_FP_v4.zip
import os; os.chdir(ROOT); print(sorted(os.listdir(ROOT)))

In [ ]:
# 2. Pustaka — versi dikunci sama dengan paper 1 (CPU-only Google Colab)
#    TensorFlow 2.20.0, scikit-learn 1.6.1, XGBoost 3.1.2, LightGBM 4.6.0, CatBoost 1.2.8
!pip -q install tensorflow==2.20.0 scikit-learn==1.6.1 xgboost==3.1.2 lightgbm==4.6.0 catboost==1.2.8 pyarrow statsmodels patsy

In [ ]:
# 2b. Cek lingkungan = paper 1 (diperiksa di interpreter baru, yang juga dipakai pipeline)
import subprocess, sys, json, os, re
probe = r'''
import sys, json, platform, os, re
import tensorflow as tf, sklearn, xgboost, lightgbm, catboost, numpy, pandas
cpu = re.search(r"model name\s*:\s*(.*)", open("/proc/cpuinfo").read())
print(json.dumps({"python": sys.version.split()[0], "tensorflow": tf.__version__, "scikit_learn": sklearn.__version__,
  "xgboost": xgboost.__version__, "lightgbm": lightgbm.__version__, "catboost": catboost.__version__,
  "numpy": numpy.__version__, "pandas": pandas.__version__, "logical_cpus": os.cpu_count(),
  "cpu_model": cpu.group(1) if cpu else "unknown", "gpus": len(tf.config.list_physical_devices("GPU"))}))
'''
out = subprocess.run([sys.executable, '-c', probe], capture_output=True, text=True)
env = json.loads(out.stdout.strip().splitlines()[-1]); print(json.dumps(env, indent=1))
REQUIRED = {'tensorflow': '2.20.0', 'scikit_learn': '1.6.1', 'xgboost': '3.1.2', 'lightgbm': '4.6.0', 'catboost': '1.2.8'}
bad = {k: env[k] for k, v in REQUIRED.items() if env[k] != v}
assert not bad, f'Versi pustaka berbeda dari paper 1: {bad} — jalankan ulang sel 2.'
assert env['python'].startswith('3.12.'), 'Python bukan 3.12 seperti paper 1.'
assert env['gpus'] == 0, 'Runtime memakai GPU; pilih Runtime → Ubah jenis runtime → CPU.'
if env['python'] != '3.12.13': print(f"Catatan: Python {env['python']} (paper 1: 3.12.13); versi patch dicatat di environment_metadata.")
if env['logical_cpus'] != 2: print(f"Catatan: {env['logical_cpus']} CPU logis (paper 1: 2).")
print('✓ Lingkungan sesuai paper 1 (CPU-only, TF 2.20.0, sklearn 1.6.1, XGB 3.1.2, LGBM 4.6.0, CatBoost 1.2.8).')

In [ ]:
# 3. Verifikasi isi paket (SHA-256 harus sama dengan versi yang diaudit)
import hashlib, subprocess, sys, json, time, pathlib
EXPECTED = {
  'code/mce_ilcadt_fp_v4.py': '7e051fb18c02c23f99b2e0cfefd97885571ff750f79f83ccb8a68f47ce52d162',
  'code/calibration_per_class.py': '8590bac0f11616c803d21a13435487feadce8b3308fa604ffeae7fa82997b90e',
  'data/master_v4_pseudonymised.parquet': 'b6e73b23911ddd02c9d04b5b390fa0745d9a41c1be55efd676496644815b2a1a',
  'SAP_v4.md': '3ec9bbe2be7108641f07be531cee25e6eedd8b55d5cde9ddf9f056db592776a6',
}
sha = lambda p: hashlib.sha256(open(p,'rb').read()).hexdigest()
for p, h in EXPECTED.items():
    assert sha(f'{ROOT}/{p}') == h, f'BERBEDA: {p}'
print('✓ Kode, data, dan SAP sesuai versi yang diaudit.')

RUNS = f'{ROOT}/runs'
CONFIGS = {  # satu perubahan per analisis sensitivitas (SAP bagian 8)
  'primary': {},
  'sens_who_label': {'MCE_FP_LABEL_SOURCE': 'who'},
  'sens_no_current_status': {'MCE_FP_ABLATION': 'no_current_status'},
  'sens_exclude_flagged': {'MCE_FP_EXCLUDE_FLAGGED': '1'},
}

def _call(config, stages, horizons):
    env = {**os.environ, **CONFIGS[config],
           'MCE_FP_MASTER_PATH': f'{ROOT}/data/master_v4_pseudonymised.parquet',
           'MCE_FP_SAP_PATH': f'{ROOT}/SAP_v4.md',
           'MCE_FP_BASE_DIR': f'{RUNS}/{config}',
           'MCE_FP_STAGES': stages, 'MCE_FP_HORIZONS': horizons}
    log = pathlib.Path(f'{RUNS}/{config}/logs'); log.mkdir(parents=True, exist_ok=True)
    logfile = log / f"{time.strftime('%Y%m%d_%H%M%S')}_{stages.replace(',','')}_h{horizons.replace(',','-')}.log"
    t = time.time()
    with open(logfile, 'w') as f:
        r = subprocess.run([sys.executable, f'{ROOT}/code/mce_ilcadt_fp_v4.py'], env=env, stdout=f, stderr=subprocess.STDOUT)
    lines = [l for l in open(logfile) if l.startswith(('✓', '  h=', 'Traceback', 'RuntimeError', 'ValueError'))]
    print(''.join(lines[-12:]) + f'   ({(time.time()-t)/60:.1f} menit; log: {logfile.name})')
    if r.returncode != 0:
        raise RuntimeError(f'Gagal — lihat {logfile}')

def status(config, h):
    art = pathlib.Path(f'{RUNS}/{config}/h{h}/artifacts')
    if (art / 'stage_f_manifest.json').exists(): return 'selesai'
    if (art / 'freeze_manifest.json').exists(): return 'dibekukan'
    if (art / 'stage_b_manifest.json').exists(): return 'pasangan siap'
    return 'belum'

def run(config, horizons, b_horizons=None):
    """Melanjutkan dari tahap terakhir yang aman untuk setiap horizon."""
    b_horizons = b_horizons or horizons
    if any(status(config, h) == 'belum' for h in horizons):
        _call(config, 'a,b', ','.join(map(str, b_horizons)))
    for h in horizons:
        s = status(config, h)
        if s == 'selesai':
            print(f'{config} h={h}: sudah selesai, dilewati'); continue
        stages = 'a,f' if s == 'dibekukan' else 'a,c,d,e,f'  # (a) mencatat versi pustaka sesi ini
        print(f'{config} h={h}: menjalankan tahap {stages} …')
        _call(config, stages, str(h))
print('Fungsi siap.')

### S1 — Analisis utama: pasangan h = 1–6, lalu model h = 1

In [ ]:
run('primary', [1], b_horizons=[1, 2, 3, 4, 5, 6])

### S2 — Analisis utama h = 3 dan h = 6

In [ ]:
run('primary', [3, 6])

### S3 — Kurva horizon (analisis tambahan): h = 2, 4, 5

In [ ]:
run('primary', [2, 4, 5])

### S4 — Sensitivitas 1: label WHO (h = 1, 3, 6)

In [ ]:
run('sens_who_label', [1, 3, 6])

### S5 — Sensitivitas 2: tanpa status bulan t (h = 1, 3, 6)

In [ ]:
run('sens_no_current_status', [1, 3, 6])

### S6 — Sensitivitas 3: tanpa observasi bertanda penurunan tinggi > 1,5 cm (h = 1, 3, 6)

In [ ]:
run('sens_exclude_flagged', [1, 3, 6])

### S7 — Kalibrasi per kelas Meta-DNN (SAP 7.3), setelah S1–S3 selesai

In [ ]:
r = subprocess.run([sys.executable, f'{ROOT}/code/calibration_per_class.py', '--run-root', f'{RUNS}/primary',
                    '--horizons', '1,2,3,4,5,6', '--out', f'{RUNS}/primary/calibration'],
                   stdout=open(f'{RUNS}/primary/logs/calibration.log', 'w'), stderr=subprocess.STDOUT)
print('✓ Kalibrasi tersimpan di runs/primary/calibration (tidak ditampilkan).' if r.returncode == 0 else 'Gagal — lihat runs/primary/logs/calibration.log')

### Status semua run dan FREEZE_LOG_v4

In [ ]:
import pandas as pd
plan = {'primary': [1,2,3,4,5,6], 'sens_who_label': [1,3,6], 'sens_no_current_status': [1,3,6], 'sens_exclude_flagged': [1,3,6]}
display(pd.DataFrame([{'analisis': c, 'h': h, 'status': status(c, h)} for c, hs in plan.items() for h in hs]))
for c in plan:
    p = f'{RUNS}/{c}/shared/FREEZE_LOG_v4.txt'
    if os.path.exists(p): print(open(p).read())

### Paket hasil untuk diunggah ke chat (CSV/JSON/log saja; tanpa model, pasangan, dan probabilitas mentah)

In [ ]:
import zipfile
out = f'{ROOT}/MCE_FP_v4_results.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in pathlib.Path(RUNS).rglob('*'):
        if p.is_file() and p.suffix in ('.csv', '.json', '.txt', '.log') and 'base_fold_bundles' not in p.parts:
            z.write(p, p.relative_to(ROOT))
print(out, round(os.path.getsize(out)/1e6, 1), 'MB')

### S8 — Run ulang independen (SAP 11.4), di runtime baru setelah S1–S2
Menjalankan ulang analisis utama h = 1, 3, 6 ke folder terpisah lalu membandingkan threshold, probabilitas, dan keputusan.

In [ ]:
CONFIGS['rerun_primary'] = {}
run('rerun_primary', [1, 3, 6])
import numpy as np
for h in (1, 3, 6):
    a, b = f'{RUNS}/primary/h{h}/artifacts', f'{RUNS}/rerun_primary/h{h}/artifacts'
    same = {f: bool(np.array_equal(np.load(f'{a}/{f}'), np.load(f'{b}/{f}')))
            for f in ('class_specific_thresholds.npy', 'meta_dnn_test_probabilities.npy', 'test_decisions.npy', 'oof_meta_dev.npy')}
    print(f'h={h}', same)